# For-Loops: Lists vs. Dictionaries — `dict.items()` Explained

**Your question, cleaned up:** Is `for key, value in dict1.items():` the same
mechanism as looping through a list (e.g. `for room, area in house.items(): print(...)`)?

**Short answer:** They're the *same syntax pattern* (`for X in iterable:`), but
they are **not** the same thing under the hood, because a **list** and a
**dictionary** hand the loop different kinds of items to unpack each round.

This notebook builds that up from the ground, term by term.

## 1. Concept: What a `for` loop actually does

A `for` loop doesn't "index through" a list or dict — it calls a low-level
protocol called **iteration**. Every loop follows this shape:

```
for <variable(s)> in <iterable>:
    <do something with variable(s)>
```

- **iterable** — any object Python knows how to hand out items from, one at a
  time (a `list`, a `dict`, a `str`, a `range`, etc.)
- On each pass, the iterable produces **one item**. What that "one item" *is*
  depends entirely on the type of iterable — that's the part people gloss
  over, and it's the actual answer to your question.

### Why it matters
If you don't know what a single "item" looks like for the iterable you're
using, you won't know how many variables to put before `in`, or what they'll
contain. This is the #1 source of `ValueError: too many values to unpack`
and `TypeError: cannot unpack non-iterable` errors for beginners.

In [1]:
# --- Iterating over a LIST ---
# A list's items are single values. Each loop pass hands you ONE value.

house_rooms = ["kitchen", "bedroom", "bathroom"]

for room in house_rooms:
    # `room` is just a string on each pass -- one variable, one value
    print(f"Room: {room}")


Room: kitchen
Room: bedroom
Room: bathroom


Notice: **one variable** (`room`) receives **one value** each pass,
because a list's items ARE single values (strings, in this case).

## 2. Concept: What a dictionary actually stores

A `dict` doesn't store loose values like a list does — it stores
**key-value pairs**. Think of it as a two-column table:

| key (room)   | value (area) |
|--------------|--------------|
| kitchen      | 12           |
| bedroom      | 15           |
| bathroom     | 6            |

When you write `dict1.items()`, you are asking the dictionary: *"give me
every row of that table, one at a time, as a `(key, value)` pair"* (a
**tuple** -- an ordered, fixed pair of two values).

So the iterable here isn't the dict itself in the raw sense -- `.items()`
is a **view** that produces `(key, value)` tuples. That's the key
(pun intended) mechanical difference from a list.

In [2]:
# --- Iterating over a DICTIONARY with .items() ---

house = {"kitchen": 12, "bedroom": 15, "bathroom": 6}

for room, area in house.items():
    # house.items() yields a TUPLE each pass: ("kitchen", 12), then ("bedroom", 15), etc.
    # Python automatically UNPACKS that 2-item tuple into (room, area)
    print(f"The {room} is {area} sqm")


The kitchen is 12 sqm
The bedroom is 15 sqm
The bathroom is 6 sqm


## 3. Direct comparison -- answering your question

```python
for room in house_rooms:            # list  -> yields ONE value per pass  (a string)
for room, area in house.items():    # dict  -> yields ONE tuple per pass  (2 values), auto-unpacked
```

**They are the same *pattern*** (`for ... in ...:`), because both lists and
`dict.items()` are iterables. But they are **not the same mechanism**:

- List: 1 loop variable <- 1 value per item
- `dict.items()`: 2 loop variables <- 1 tuple (containing 2 values) per item, unpacked automatically

If you tried `for room, area in house_rooms:` (looping the *list* with two
variables), Python would try to unpack each **string** into two characters
and mostly error out or give nonsense -- because a plain list of strings
never hands you 2-value tuples.

Run the cell below to see that failure mode directly (this is "get actual
feedback, not predicted feedback" -- let's watch the real error rather than
just describe it).

In [3]:
# Proof: what happens if you try to unpack a LIST like a dict.items()?

house_rooms = ["kitchen", "bedroom", "bathroom"]

try:
    for room, area in house_rooms:   # WRONG on purpose: each item is a single string, not a 2-item pair
        print(room, area)
except ValueError as e:
    print(f"Real error raised: {e}")

# Why: "kitchen" is a string of 7 characters. Python tries to unpack it as if
# it were exactly 2 items, and fails because it has 7 characters, not 2.


Real error raised: too many values to unpack (expected 2)


## 4. Bonus confusion point: looping a dict *without* `.items()`

This trips up almost everyone at some stage, so it's worth seeing directly.

In [4]:
house = {"kitchen": 12, "bedroom": 15, "bathroom": 6}

# Looping a dict directly (no .items()) only gives you the KEYS
for room in house:
    print(room)          # kitchen, bedroom, bathroom -- values are NOT shown

print("---")

# To get values only, use .values()
for area in house.values():
    print(area)           # 12, 15, 6

print("---")

# To get BOTH key and value together, use .items()
for room, area in house.items():
    print(room, area)


kitchen
bedroom
bathroom
---
12
15
6
---
kitchen 12
bedroom 15
bathroom 6


## 5. Real-world example

Say you're building a small data-analyst script that reports which rooms in
a house exceed a minimum size requirement (a realistic filtering task you'll
do constantly with dataframes/dicts in analyst work).

In [5]:
house = {"kitchen": 12, "bedroom": 15, "bathroom": 6, "living_room": 22}
MIN_SIZE_SQM = 10

qualifying_rooms = []

for room, area in house.items():
    if area >= MIN_SIZE_SQM:
        qualifying_rooms.append(room)

print("Rooms at or above the minimum size:")
for room in qualifying_rooms:       # plain list loop -- one value per pass
    print(f" - {room}")


Rooms at or above the minimum size:
 - kitchen
 - bedroom
 - living_room


Notice how naturally both loop styles show up together in one real task:
the `dict.items()` loop to inspect key-value data, and the plain list loop to
report a simple collected result. Same `for ... in ...:` syntax, two
different jobs, because the two iterables hand out different shapes of data.

## 6. Common beginner mistakes with this pattern

1. **Forgetting `.items()`** and trying `for room, area in house:` on a raw
   dict -- this actually raises `ValueError: too many values to unpack`
   because looping a dict directly only yields keys (single strings), not
   pairs.
2. **Using `.items()` on a list** -- lists don't have an `.items()` method
   at all; that only exists on `dict` (and similar mapping types).
3. **Wrong variable order** -- `.items()` always yields `(key, value)` in
   that order. Writing `for area, room in house.items():` will silently
   swap the meaning of your variables without raising any error, since
   Python has no idea which name you *meant* -- it just assigns positionally.
   This is a logic bug, not a crash, so it's more dangerous.

## 7. Small challenge

Given this dictionary:

```python
inventory = {"apples": 30, "bananas": 5, "oranges": 0}
```

Write a loop using `.items()` that prints `"<item>: OUT OF STOCK"` when the
count is `0`, and `"<item>: <count> in stock"` otherwise.

Try it in the empty cell below before checking the solution underneath.

In [6]:
inventory = {"apples": 30, "bananas": 5, "oranges": 0}

# Your attempt here:



In [7]:
# --- Solution ---
inventory = {"apples": 30, "bananas": 5, "oranges": 0}

for item, count in inventory.items():
    if count == 0:
        print(f"{item}: OUT OF STOCK")
    else:
        print(f"{item}: {count} in stock")


apples: 30 in stock
bananas: 5 in stock
oranges: OUT OF STOCK
